# Food Delivery Data Analysis System

This notebook performs data collection, data cleaning, exploratory analysis, NumPy statistics, and visualizations.

**Instructions:** Run cells from top to bottom and upload your food delivery CSV when prompted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Google Colab file upload; in local Jupyter, enter the CSV path when prompted.
try:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")
    file_name = next(iter(uploaded))
except ImportError:
    file_name = input("Enter the path to your CSV file: ").strip()

df = pd.read_csv(file_name)
print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


## 1. Data Collection and Initial Checks

In [ ]:
print("BEFORE CLEANING")
print("--------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", int(df.isnull().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\\nMissing values by column:")
display(df.isnull().sum().to_frame("Missing Values"))


## 2. Data Cleaning

Duplicate rows are removed. Missing numeric values are filled with the column median; missing text/category values are filled with the most frequent value when available.

In [ ]:
df = df.drop_duplicates().copy()

for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        median_value = df[col].median()
        if pd.notna(median_value):
            df[col] = df[col].fillna(median_value)

for col in df.select_dtypes(include=["object", "category", "string"]).columns:
    if df[col].isnull().any():
        mode_values = df[col].mode(dropna=True)
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

print("AFTER CLEANING")
print("--------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values remaining:", int(df.isnull().sum().sum()))
print("Duplicate rows remaining:", int(df.duplicated().sum()))
display(df.head())


## 3. Pandas Data Analysis

In [ ]:
print("DATASET SUMMARY")
display(df.describe(include="all").T)

# Category counts, if the expected column exists
if "Food_Category" in df.columns:
    print("\\nOrders by food category:")
    display(df["Food_Category"].value_counts(dropna=False).to_frame("Order Count"))

if "Order_Status" in df.columns:
    print("\\nOrder status counts:")
    display(df["Order_Status"].value_counts(dropna=False).to_frame("Order Count"))

for col in ["Order_Amount", "Delivery_Time_Min", "Rating"]:
    if col in df.columns:
        print(f"\\nStatistics for {col}:")
        print(df[col].describe())


## 4. NumPy Statistical Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

if numeric_cols:
    print("NUMPY STATISTICS")
    for col in numeric_cols:
        values = df[col].dropna().to_numpy()
        if values.size:
            print(f"\\n{col}")
            print("Mean:", round(float(np.mean(values)), 2))
            print("Minimum:", float(np.min(values)))
            print("Maximum:", float(np.max(values)))
            print("Standard deviation:", round(float(np.std(values)), 2))
else:
    print("No numerical columns are available for NumPy analysis.")


## 5. Data Visualization

In [ ]:
# Chart 1: Number of orders by food category
if "Food_Category" in df.columns:
    category_counts = df["Food_Category"].value_counts().sort_values(ascending=False)
    if not category_counts.empty:
        category_counts.plot(kind="bar", figsize=(9, 5))
        plt.title("Number of Orders by Food Category")
        plt.xlabel("Food Category")
        plt.ylabel("Number of Orders")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("Chart 1 skipped: 'Food_Category' column not found.")

# Chart 2: Distribution of order statuses
if "Order_Status" in df.columns:
    status_counts = df["Order_Status"].value_counts()
    if not status_counts.empty:
        status_counts.plot(kind="pie", autopct="%1.1f%%", figsize=(7, 7))
        plt.title("Order Status Distribution")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()
else:
    print("Chart 2 skipped: 'Order_Status' column not found.")

# Chart 3: Average delivery time by food category
if {"Food_Category", "Delivery_Time_Min"}.issubset(df.columns):
    delivery = df[["Food_Category", "Delivery_Time_Min"]].copy()
    delivery["Delivery_Time_Min"] = pd.to_numeric(delivery["Delivery_Time_Min"], errors="coerce")
    avg_delivery = delivery.groupby("Food_Category")["Delivery_Time_Min"].mean().dropna()
    if not avg_delivery.empty:
        avg_delivery.plot(kind="line", marker="o", figsize=(9, 5))
        plt.title("Average Delivery Time by Food Category")
        plt.xlabel("Food Category")
        plt.ylabel("Average Delivery Time (minutes)")
        plt.xticks(rotation=45, ha="right")
        plt.grid(True)
        plt.tight_layout()
        plt.show()
else:
    print("Chart 3 skipped: requires 'Food_Category' and 'Delivery_Time_Min' columns.")


## 6. Conclusion

Review the actual summary statistics and charts above. Write your conclusion based on the patterns found in your uploaded dataset; do not report assumptions as results.